# Schedule Generator

---

*Runs Before*: None

*Runs After*: ``fov.ipynb``, ``skygrid.ipynb``, ``skyblocks.ipynb``

This notebook turns the scheduling blocks produced by ``skyblocks.ipynb`` into a concrete mission
timeline. It:

1. Generates the downlink schedule: the fixed times and spacecraft orientations at which UVEX
   must return to Earth-pointing for ``DOWNLINK_DURATION``, once every
   ``TIME_BETWEEN_GROUND_CONTACT``.
2. Runs a demand-vs-supply diagnostic: is the requested cadence (each block's
   ``EXPECTED_VISITS``) even achievable within the prime mission duration?
3. Determines which blocks are observable (Sun/Moon/Earth-limb constraints) at each downlink
   slot.
4. Matches blocks to downlink slots via a weighted bipartite matching, visiting each block up to
   its ``EXPECTED_VISITS`` times, preferring low-background slots.
5. For each matched (block, slot) pair, solves a TSP over that block's fields to find the
   slew-minimizing observing order, and stitches the per-block plans into a single mission
   timeline.
6. Writes the resulting plan to disk, and (optionally) renders an animation of it.

## Settings / Configuration


In [ ]:
%xmode minimal

In [ ]:
# ========================================= #
# Manage Warnings                           #
# ========================================= #
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
warnings.filterwarnings("ignore", ".*dubious year.*")
warnings.filterwarnings(
    "ignore", "Tried to get polar motions for times after IERS data is valid.*"
)

In [ ]:
# ========================================= #
# Imports                                   #
# ========================================= #
from pathlib import Path

import networkx as nx
import numpy as np
import synphot
from astropy import units as u
from astropy.coordinates import SkyCoord, UnitSphericalRepresentation
from astropy.table import QTable, vstack
from astropy.time import Time
from astropy.utils.masked import Masked, combine_masks
from ligo.skymap import plot  # noqa: F401
from m4opt import fov
from m4opt.dynamics import EigenAxisSlew, nominal_roll
from m4opt.fov._core import circle_to_polygon
from m4opt.missions import uvex as mission
from m4opt.missions import uvex_downlink_orientation
from m4opt.synphot import observing
from m4opt.utils.optimization import solve_tsp
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.collections import PatchCollection
from regions import Regions
from tqdm.auto import tqdm

In [ ]:
# ========================================= #
# ``%%skipif`` cell magic                   #
# ========================================= #
# Lets a cell be gated behind a boolean setting (e.g. ``SAVE_SCHEDULE_ANIMATION``) without
# commenting it out. The expression on the magic line is evaluated against the notebook
# namespace; if it is True, the rest of the cell is not executed. Usage: ``%%skipif <expr>`` as
# the first line of a cell.
from IPython import get_ipython
from IPython.core.magic import register_cell_magic


@register_cell_magic
def skipif(line, cell):
    # Evaluate the expression given to the magic command
    if eval(line, get_ipython().user_ns):
        print(f"Cell skipped: condition '{line}' is True.")
        return
    # If False, execute the cell code normally
    get_ipython().run_cell(cell)

### Configuration Settings

A number of configuration settings are available in this notebook. Please configure the options below to your liking. The default settings are recommended for most users.

In [ ]:
# ========================================= #
# Settings and Configuration                #
# ========================================= #

# --- CORE SETTINGS --- #
# OUTPUT_PATH: Directory shared with ``fov.ipynb``, ``skygrid.ipynb``, and ``skyblocks.ipynb``.
#              This notebook reads their outputs from here and writes the final schedule (and any
#              plots/animation) back into it.
OUTPUT_PATH = Path("../Runs/TEST")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# --- FOV SETTINGS --- #
# Paths to the DS9 region files written by ``fov.ipynb``. If ``None``, they are looked up in
# ``OUTPUT_PATH`` (i.e. wherever ``fov.ipynb`` wrote them).
MISSION_FOV_PATH = None
INSCRIBED_CIRCLE_FOV_PATH = None

# --- BLOCK SETTINGS --- #
# Path to the fields/blocks table written by ``skyblocks.ipynb``. If ``None``, it is looked up in
# ``OUTPUT_PATH`` (i.e. wherever ``skyblocks.ipynb`` wrote it).
BLOCK_FILE_PATH = None

# --- UVEX MISSION SETTINGS --- #
TIME_BETWEEN_GROUND_CONTACT = 6 * u.hour
DOWNLINK_DURATION = 30 * u.minute
DWELL_DURATION = 900 * u.second
PRIME_MISSION_DURATION = 2 * u.year
MISSION_START_TIME = Time("2030-01-01")

# --- SCHEDULING SETTINGS --- #
# SNR_TARGET: Target signal-to-noise ratio used to estimate the FUV exposure time that drives the
#             per-slot background weighting in the matching step below.
SNR_TARGET = 5

# --- ANIMATION SETTINGS --- #
# Controls for the optional survey-schedule animation generated at the end of this notebook.
# Animating the full PRIME_MISSION_DURATION timeline is expensive, so this is off by default --
# set SAVE_SCHEDULE_ANIMATION = True and tune ANIMATION_N_FRAMES to render it.
SAVE_SCHEDULE_ANIMATION = False
ANIMATION_N_FRAMES = 20
ANIMATION_DPI = 150
ANIMATION_INTERVAL = 300  # milliseconds between frames

## Field of View and Blocks

Load the FOV region files produced by ``fov.ipynb`` and the fields/blocks table produced by
``skyblocks.ipynb``.

In [ ]:
# ========================================= #
# Load FOV Files and the Blocks Table       #
# ========================================= #
MISSION_FOV_PATH = (
    OUTPUT_PATH / "chips.ds9" if MISSION_FOV_PATH is None else Path(MISSION_FOV_PATH)
)
INSCRIBED_CIRCLE_FOV_PATH = (
    OUTPUT_PATH / "inscribed-circle.ds9"
    if INSCRIBED_CIRCLE_FOV_PATH is None
    else Path(INSCRIBED_CIRCLE_FOV_PATH)
)
BLOCK_FILE_PATH = (
    OUTPUT_PATH / "fields.ecsv" if BLOCK_FILE_PATH is None else Path(BLOCK_FILE_PATH)
)

for _path, _source_notebook in (
    (MISSION_FOV_PATH, "fov.ipynb"),
    (INSCRIBED_CIRCLE_FOV_PATH, "fov.ipynb"),
    (BLOCK_FILE_PATH, "skyblocks.ipynb"),
):
    if not _path.exists():
        raise FileNotFoundError(
            f"File not found at {_path}. Run {_source_notebook} first (writing to the same OUTPUT_PATH) to generate it."
        )

MISSION_FOV = Regions.read(str(MISSION_FOV_PATH))
(INSCRIBED_CIRCLE_FOV,) = Regions.read(str(INSCRIBED_CIRCLE_FOV_PATH))

BLOCKS = QTable.read(BLOCK_FILE_PATH)
BLOCK_IDS = BLOCKS["BLOCK_ID"]

## Downlink Schedule

In every ``TIME_BETWEEN_GROUND_CONTACT`` observing window, the telescope must return to its
downlinking orientation for ``DOWNLINK_DURATION``. Because these are entirely known in the
schedule *a priori*, we can generate the slots for these up front.

In [ ]:
# ========================================= #
# Generate Downlink Times and Orientations  #
# ========================================= #
DOWNLINK_TIMES = (
    MISSION_START_TIME
    + np.arange(
        0,
        PRIME_MISSION_DURATION.to_value(u.day),
        TIME_BETWEEN_GROUND_CONTACT.to_value(u.day),
    )
    * u.day
)

DOWNLINK_TARGET_COORDS, DOWNLINK_ROLLS = uvex_downlink_orientation(DOWNLINK_TIMES)
DOWNLINK_TARGET_COORDS = DOWNLINK_TARGET_COORDS.icrs
DOWNLINK_OBSERVER_LOCATIONS = mission.observer_location(DOWNLINK_TIMES)

DOWNLINK_TABLE = QTable(
    {
        "start_time": DOWNLINK_TIMES,
        "target_coord": SkyCoord(
            DOWNLINK_TARGET_COORDS.ra,
            DOWNLINK_TARGET_COORDS.dec,
            representation_type=UnitSphericalRepresentation,
        ),
        "roll": DOWNLINK_ROLLS,
        "observer_location": DOWNLINK_OBSERVER_LOCATIONS,
    }
)
DOWNLINK_TABLE["action"] = "downlink"
DOWNLINK_TABLE["duration"] = DOWNLINK_DURATION
DOWNLINK_TABLE.write(OUTPUT_PATH / "downlinks.ecsv", overwrite=True)

## Demand vs. Supply Diagnostic

Before running the (expensive) matching and TSP steps below, sanity-check whether the requested
cadence is even achievable. Each block-visit consumes one full ground-contact window -- downlink,
slews, and dwells all fit within ``TIME_BETWEEN_GROUND_CONTACT`` by construction (see
``MAX_DWELLS_PER_BLOCK`` in ``skyblocks.ipynb``) -- so the total time needed to deliver every
block's ``EXPECTED_VISITS`` is simply the total number of block-visits times
``TIME_BETWEEN_GROUND_CONTACT``. If that exceeds ``PRIME_MISSION_DURATION``, the matching below is
guaranteed to leave some blocks under-visited.

In [ ]:
# ========================================= #
# Summarize Block-Visit Demand by Category  #
# ========================================= #
BLOCK_SUMMARY = (
    BLOCKS["BLOCK_ID", "EXPECTED_VISITS"].group_by("BLOCK_ID").groups.aggregate(np.max)
)
BLOCK_SUMMARY.sort("BLOCK_ID")
BLOCK_VISIT_MULTIPLICITY = BLOCK_SUMMARY["EXPECTED_VISITS"]

# A block is homogeneous in region membership by construction (skyblocks.ipynb partitions one
# cadence tier at a time), so tagging a block by whether ANY of its fields carry a region flag
# recovers each block's category.
BLOCK_CATEGORIES = (
    BLOCKS[
        "BLOCK_ID",
        "IN_ALL_SKY_BASE",
        "IN_LMLZ_WIDE",
        "IN_LMLZ_DEEP",
        "IN_LMLZ_DEEP_POINTS",
        "IN_MC",
    ]
    .group_by("BLOCK_ID")
    .groups.aggregate(np.any)
)
BLOCK_CATEGORIES.sort("BLOCK_ID")

CATEGORY_COLUMNS = {
    "ALL SKY BASE": "IN_ALL_SKY_BASE",
    "LMLZ WIDE": "IN_LMLZ_WIDE",
    "LMLZ DEEP": "IN_LMLZ_DEEP",
    "LMLZ DEEP POINTS": "IN_LMLZ_DEEP_POINTS",
    "MC": "IN_MC",
}

print("-----------------------------------------------------------------")
print(f"TOTAL BLOCKS: {len(BLOCK_SUMMARY)}")
for label, column in CATEGORY_COLUMNS.items():
    _in_category = BLOCK_CATEGORIES[column]
    _n_blocks = np.sum(_in_category)
    _n_visits = BLOCK_VISIT_MULTIPLICITY[_in_category].sum()
    _time_needed = (_n_visits * TIME_BETWEEN_GROUND_CONTACT).to(u.day)
    print(
        f"BLOCKS IN {label}: {_n_blocks} ({_n_blocks / len(BLOCK_SUMMARY) * 100:.2f}%)"
    )
    print(f"\tBlock-Visits Needed: {_n_visits:.0f}")
    print(f"\tTime Needed (incl. downlink/slew): {_time_needed:.2f}")
print("=================================================================")
TOTAL_VISITS_NEEDED = BLOCK_VISIT_MULTIPLICITY.sum()
TOTAL_TIME_NEEDED = (TOTAL_VISITS_NEEDED * TIME_BETWEEN_GROUND_CONTACT).to(u.day)
print(f"Total Block-Visits Needed: {TOTAL_VISITS_NEEDED:.0f}")
print(f"Total Time Needed (incl. downlink/slew): {TOTAL_TIME_NEEDED:.2f}")
print(f"Prime Mission Duration: {PRIME_MISSION_DURATION.to(u.day):.2f}")
print("-----------------------------------------------------------------")

if TOTAL_TIME_NEEDED > PRIME_MISSION_DURATION:
    _overage = TOTAL_TIME_NEEDED - PRIME_MISSION_DURATION
    _pct_of_budget = (TOTAL_TIME_NEEDED / PRIME_MISSION_DURATION).to_value(
        u.dimensionless_unscaled
    ) * 100
    print("===================================================================")
    print("WARNING: requested cadence exceeds the prime mission duration.")
    print(f"  Needed:  {TOTAL_TIME_NEEDED:.2f} ({_pct_of_budget:.1f}% of budget)")
    print(f"  Have:    {PRIME_MISSION_DURATION.to(u.day):.2f}")
    print(f"  Overage: {_overage:.2f}")
    print("  Not every block will receive its expected number of visits.")
    print("===================================================================")

## Determine Fully Observable Blocks

As part of the scheduling constraints for the mission, we can only perform observations which are
suitably visible when considering the position of the Sun, Moon, and Earth. As such, we need to
determine, for each block, which downlink slots it is observable at.

In [ ]:
# ========================================= #
# Construct Block Times                     #
# ========================================= #
# The block times are the bin centers for each observing block, including the time allocated to
# downlinking.
ALLOCATED_BLOCK_TIMES = DOWNLINK_TIMES[:-1] + TIME_BETWEEN_GROUND_CONTACT / 2
ALLOCATED_BLOCK_OBSERVER_LOCATIONS = mission.observer_location(ALLOCATED_BLOCK_TIMES)

ALLOCATED_BLOCK_MISSION_CONSTRAINTS = mission.constraints(
    ALLOCATED_BLOCK_OBSERVER_LOCATIONS[np.newaxis, :],
    BLOCKS["TARGET_COORD"][:, np.newaxis],
    ALLOCATED_BLOCK_TIMES[np.newaxis, :],
)

# An (N_BLOCKS, N_SLOTS) array: True where every field in a block is simultaneously observable at
# that slot.
OBSERVABILITY_GRID = np.asarray(
    [
        np.logical_and.reduce(
            ALLOCATED_BLOCK_MISSION_CONSTRAINTS[BLOCK_IDS == i, :], axis=0
        )
        for i in range(BLOCK_IDS.max() + 1)
    ]
)

## Determine Weights

Calculate a per-block, per-slot weight equal to the total exposure time required to reach
``SNR_TARGET`` for every field in the block, at that slot's sky background. Lower weight (less
background-limited exposure time needed) is preferred by the matching step below.

In [ ]:
# ========================================= #
# Compute the Exposure-Time Weight Grid     #
# ========================================= #
with observing(
    ALLOCATED_BLOCK_OBSERVER_LOCATIONS[np.newaxis, :],
    BLOCKS["TARGET_COORD"][:, np.newaxis],
    ALLOCATED_BLOCK_TIMES[np.newaxis, :],
):
    _exptime = mission.detector.get_exptime(
        SNR_TARGET,
        synphot.SourceSpectrum(synphot.ConstFlux1D, amplitude=24.5 * u.ABmag),
        "FUV",
    ).to_value(u.s)

EXPOSURE_WEIGHT_GRID = np.asarray(
    [np.sum(_exptime[BLOCK_IDS == i, :], axis=0) for i in range(BLOCK_IDS.max() + 1)]
)
EXPOSURE_WEIGHT_GRID = np.clip(EXPOSURE_WEIGHT_GRID, 0, 86400)

## Optimize

Match each block to up to ``EXPECTED_VISITS`` downlink slots via a weighted bipartite matching:
for every (block, slot) pair where the block is observable, add an edge weighted by
``EXPOSURE_WEIGHT_GRID``, replicated once per remaining expected visit. The minimum-weight full
matching then assigns each slot to the block/visit that needs it least expensively, while
respecting each block's total visit budget.

In [ ]:
# ========================================= #
# Perform the Weighted Bipartite Matching   #
# ========================================= #
MATCHING_GRAPH = nx.Graph()
MATCHING_GRAPH.add_weighted_edges_from(
    ((i, visit), j, EXPOSURE_WEIGHT_GRID[i, j])
    for i, j in zip(*np.nonzero(OBSERVABILITY_GRID))
    for visit in range(int(BLOCK_VISIT_MULTIPLICITY[i]))
)

MATCHING = nx.bipartite.minimum_weight_full_matching(
    MATCHING_GRAPH, np.arange(OBSERVABILITY_GRID.shape[1])
)

MATCHING_I, MATCHING_J = np.transpose(
    [(MATCHING[j][0], j) for j in range(OBSERVABILITY_GRID.shape[1]) if j in MATCHING]
)

In [ ]:
# ========================================= #
# Construct the Observation Target List     #
# ========================================= #
TARGETS = QTable(
    {
        "field_id": BLOCKS["FIELD_ID"],
        "block_id": BLOCKS["BLOCK_ID"],
        "target_coord": SkyCoord(
            BLOCKS["TARGET_COORD"].ra,
            BLOCKS["TARGET_COORD"].dec,
            representation_type=UnitSphericalRepresentation,
        ),
    }
)
TARGETS["action"] = "observe"
TARGETS["duration"] = DWELL_DURATION
TARGETS

## Sequence Each Block via TSP

For every matched (block, slot) pair, interleave the block's targets with the downlink pointing
that starts it and solve a TSP for the slew-minimizing observing order, then stitch the per-block
plans together (in slot order) into the full mission timeline.

Each block's plan keeps every one of its dwells; when two block-visits are stitched back to back,
only the trailing row of the earlier one is dropped, since the next block's downlink pointing
immediately supersedes it.

In [ ]:
# ========================================= #
# Sequence and Stitch the Mission Timeline  #
# ========================================= #
def plan_block(downlink, targets):
    """
    Sequence one scheduling block: interleave its targets with the downlink pointing and solve
    the TSP for the slew order that minimizes total slew time.

    Parameters
    ----------
    downlink : astropy.table.QTable
        A single-row table for the downlink pointing that starts the block.
    targets : astropy.table.QTable
        The block's observation targets.

    Returns
    -------
    astropy.table.QTable
        The block's plan: downlink and observations interleaved with the slews between them, in
        chronological order.
    """
    slew_targets = vstack((downlink, targets))
    slew_targets["roll"] = nominal_roll(
        slew_targets["observer_location"][0],
        slew_targets["target_coord"],
        slew_targets["start_time"][0],
    )

    slew_time = mission.slew.time(
        slew_targets["target_coord"][:, np.newaxis],
        slew_targets["target_coord"][np.newaxis, :],
        slew_targets["roll"][:, np.newaxis],
        slew_targets["roll"][np.newaxis, :],
    )
    seq, _ = solve_tsp(slew_time.to_value(u.s), verbose=False)
    assert seq[0] == 0

    # Find optimal slew path.
    slews = QTable({"duration": slew_time[seq[:-1], seq[1:]]})
    slews["action"] = "slew"

    # Interleave slews with observations.
    slew_targets = slew_targets[seq]
    slew_targets["i"] = np.arange(len(slew_targets))
    slews["i"] = np.arange(len(slews)) + 0.5
    plan = vstack((slew_targets, slews))
    plan.sort("i")
    del plan["i"]

    # Fill in observation times, observer locations.
    plan["start_time"][1:] = plan["start_time"][0] + np.cumsum(plan["duration"][:-1])
    plan["observer_location"] = mission.observer_location(plan["start_time"])

    return plan


PLAN = vstack(
    [
        plan_block(DOWNLINK_TABLE[j : j + 1], TARGETS[BLOCK_IDS == i])[
            : (-1 if n < len(MATCHING_J) - 1 else None)
        ]
        for n, (i, j) in enumerate(zip(tqdm(MATCHING_I), MATCHING_J))
    ]
)
PLAN

Sanity check: no actions overlap in time.

In [ ]:
# ========================================= #
# Sanity Check: No Overlapping Actions      #
# ========================================= #
_end_time = PLAN["start_time"] + PLAN["duration"]
_min_gap = ((PLAN["start_time"][1:] - _end_time[:-1]).to(u.s)).min()

print("-----------------------------------------------------------------")
print(f"Minimum gap between consecutive actions: {_min_gap:.2e}")
print("-----------------------------------------------------------------")
assert _min_gap >= -1e-3 * u.s

Add total time metadata.

In [ ]:
# ========================================= #
# Add Total Time Metadata                   #
# ========================================= #
_total_time_by_action = (
    PLAN["action", "duration"].group_by("action").groups.aggregate(np.sum)
)
PLAN.meta["total_time"] = {
    str(row["action"]): row["duration"] for row in _total_time_by_action
}

_end_time = PLAN["start_time"] + PLAN["duration"]
_down_time = (PLAN["start_time"][1:] - _end_time[:-1]).to(u.s)
PLAN.meta["total_time"]["slack"] = np.maximum(_down_time, 0 * u.s).sum()

Add slew angles.

In [ ]:
# ========================================= #
# Add Slew Angles                           #
# ========================================= #
_args = [
    PLAN["target_coord"][:-2],
    PLAN["target_coord"][2:],
    PLAN["roll"][:-2],
    PLAN["roll"][2:],
]
PLAN["slew_angle"] = Masked(
    np.pad(EigenAxisSlew.separation(*_args), 1, constant_values=np.nan),
    np.pad(combine_masks([arg.mask for arg in _args]), 1, constant_values=True),
)

## Save Schedule to Disk

Write the completed mission timeline -- downlinks, slews, and observations in chronological order
-- to ``initial-survey.ecsv``.

In [ ]:
# ========================================= #
# Save the Schedule to Disk                 #
# ========================================= #
PLAN[
    "start_time",
    "duration",
    "observer_location",
    "action",
    "target_coord",
    "roll",
    "field_id",
    "block_id",
    "slew_angle",
].write(OUTPUT_PATH / "initial-survey.ecsv", overwrite=True)

## Animation

Render an (optional) animation of the survey schedule: for each field, the fraction of its
expected cadence delivered so far. Controlled by ``SAVE_SCHEDULE_ANIMATION`` -- off by default,
since animating the full ``PRIME_MISSION_DURATION`` timeline is expensive.

In [ ]:
%%skipif not SAVE_SCHEDULE_ANIMATION
# ========================================= #
# Animate the Survey Schedule               #
# ========================================= #
# Render an all-sky animation of the mission timeline: one patch per field, colored (Blues) by
# how much of that field's block cadence has been delivered so far -- 0 (white) means never
# visited, 1 (deep blue) means fully done -- plus a running day counter.
_animation_total_duration = PLAN["start_time"][-1] + PLAN["duration"][-1] - PLAN["start_time"][0]
_animation_frame_times = PLAN["start_time"][0] + np.linspace(0, 1, ANIMATION_N_FRAMES) * _animation_total_duration

# Each matched block-visit actually happens at the start of its assigned downlink slot
# (MATCHING_I/MATCHING_J from the Optimize section above), independent of the TSP ordering of
# fields within the block.
_visit_block_ids = MATCHING_I
_visit_start_times = DOWNLINK_TABLE["start_time"][MATCHING_J]
_n_blocks = BLOCK_IDS.max() + 1


def _visits_so_far(time):
    return np.bincount(_visit_block_ids[_visit_start_times < time], minlength=_n_blocks)


_shift = 4 * u.hourangle
_center = SkyCoord(12 * u.hourangle - _shift, 0 * u.deg)

fig = plt.figure(dpi=ANIMATION_DPI)
ax = fig.add_subplot(projection="astro aitoff", center=_center)
for key in ["ra", "dec"]:
    ax.coords[key].set_ticklabel_visible(False)
    ax.coords[key].set_ticks_visible(False)
_transform = ax.get_transform("world")

# Build one patch per field (at a fixed position) up front; per frame we only ever update the
# color array, never the geometry.
_field_polygons = []
_field_patch_block_ids = []
for block_id, region in zip(BLOCKS["BLOCK_ID"], fov.footprint(INSCRIBED_CIRCLE_FOV, BLOCKS["TARGET_COORD"])):
    vertices = circle_to_polygon(region, 16).vertices
    for cut_vertices in plot.cut_prime_meridian(
        np.column_stack(((vertices.ra + _shift).to_value(u.rad), vertices.dec.rad))
    ):
        cut_vertices[:, 0] -= _shift.to_value(u.rad)
        _field_polygons.append(plt.Polygon(np.rad2deg(cut_vertices), closed=True))
        _field_patch_block_ids.append(block_id)
_field_patch_block_ids = np.asarray(_field_patch_block_ids)

_field_patches = PatchCollection(_field_polygons, transform=_transform, cmap="Blues", edgecolor="none")
_field_patches.set_clim(0, BLOCK_VISIT_MULTIPLICITY.max())
_field_patches.set_array(np.zeros(len(_field_polygons)))
ax.add_collection(_field_patches)

_day_text = fig.text(0.5, 0.95, "", ha="center", va="top", fontsize="x-large", fontweight="bold")

with tqdm(total=len(_animation_frame_times)) as progress:

    def animate(i):
        time = _animation_frame_times[i]
        _field_patches.set_array(_visits_so_far(time)[_field_patch_block_ids])
        _day_text.set_text(f"Day {(time - _animation_frame_times[0]).to_value(u.day):.1f}")
        progress.update()
        return [_field_patches, _day_text]

    FuncAnimation(
        fig, animate, np.arange(len(_animation_frame_times)), blit=True, interval=ANIMATION_INTERVAL
    ).save(OUTPUT_PATH / "initial-survey.mp4")
